In [42]:
from pathlib import Path

import pandas as pd

In [43]:
### Directories
project_root = Path.cwd()

ground_truth_directory = project_root / Path("eval/gt")
predicted_directory = project_root / Path("eval/pred")
result_directory = project_root / Path("eval/result")

ground_truth_directory.mkdir(parents=True, exist_ok=True)
predicted_directory.mkdir(parents=True, exist_ok=True)
result_directory.mkdir(parents=True, exist_ok=True)

print(f"Current Working Directory: {project_root}")

Current Working Directory: c:\Gabriel_Files\Programming_Files\School\Thesis\src


In [44]:
### Clear Existing Output Directories
for item in result_directory.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

In [45]:
### Main Function
def evaluate_result(truth_csv_path, predicted_csv_path):
    
    ### Initialize Dataframes
    df_truth = pd.read_csv(truth_csv_path)
    df_pred = pd.read_csv(predicted_csv_path)


    ### Clean Up Dataframes
    df_truth.columns = df_truth.columns.str.strip()
    df_pred.columns = df_pred.columns.str.strip()


    ### Merge Both Dataframes
    merged = pd.merge(df_truth, df_pred, on=['frame_index', 'human_id', 'object_id'], how='outer', indicator=True)


    ### Checks for True Positives, False Positives, and False Negatives From The Merge DataFrame 
    tp = (merged['_merge'] == 'both').sum()
    fp = (merged['_merge'] == 'right_only').sum()
    fn = (merged['_merge'] == 'left_only').sum()
    # print(f"True Positives: {tp}")
    # print(f"False Positives: {tp}")
    # print(f"True Negatives: {tp}")


    ### Calculate Precision, Recall, and F1
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * ((precision * recall) / (precision + recall))

    return precision, recall, f1


In [46]:
gt_file = ground_truth_directory / "vid17.csv"
pred_file = predicted_directory / "vid17.csv"

precision, recall, f1 = evaluate_result(gt_file, pred_file)

print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1: {f1}")

Precision: 1.0
Recall: 1.0
F1: 1.0


In [47]:
# df_true = pd.read_csv(gt_csv, header=None, names=['frame_index', 'human_Id', 'object_id'])
# df_pred = pd.read_csv(pred_csv, header=None, names=['frame_index', 'human_Id', 'object_id'])

# tp = len(df_true.merge(df_pred, on=['frame_index', 'human_Id', 'object_id'], how='inner'))
# fp = len(df_pred) - tp
# fn = len(df_true) - tp

# precision = tp / (tp + fp) if (tp + fp) > 0 else 0
# recall = tp / (tp + fn) if (tp + fn) > 0 else 0
# f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# print(f"Precision: {precision}")
# print(f"Recall: {recall}")
# print(f"F1: {f1}")